In [1]:
import os, sys, time
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.append("..")

import pandas as pd
from src.chains import build_llm, SMALL_MODEL
from src.tools import search_technique, search_video, search_wikipedia, ibjjf_points, query_adcc
from src.verifier import verify, contexts_from_tool_output

llm = build_llm()
llm_small = build_llm(SMALL_MODEL)
pd.set_option("display.max_colwidth", 80)


## 5.1–5.2 Verificador NLI e coerência lógica

Cross-encoder NLI multilíngue (mDeBERTa-v3 XNLI) compara cada afirmação da resposta (PT) com os trechos
recuperados (EN): sustentada / neutra / contradiz. A coerência compara as afirmações entre si.
Teste controlado: 4 respostas escritas à mão sobre o mesmo contexto real.


In [2]:
contexts = contexts_from_tool_output(search_technique.invoke("armlock closed guard steps"))
print(len(contexts), "trechos recuperados\n")

TEST_ANSWERS = {
    "fiel": "Segure a manga ou o punho do oponente e puxe o braço dele sobre o seu peito. "
            "Passe a perna oposta por cima da cabeça do oponente e aperte os joelhos. "
            "Eleve o quadril devagar até o oponente bater.",
    "contradiz_fonte": "Solte o punho do oponente antes de elevar o quadril. "
                       "Aplique a chave rápido e com força, sem esperar o oponente bater.",
    "alucinacao": "O armlock foi criado por Helio Gracie em 1930. "
                  "É a finalização mais usada da história do ADCC.",
    "incoerente": "Eleve o quadril devagar para finalizar a chave de braço. "
                  "Nunca eleve o quadril durante a chave de braço.",
}

t0 = time.perf_counter()
for name, ans in TEST_ANSWERS.items():
    r = verify(ans, contexts)
    print(f"[{name}] veredito={r['verdict']} | suporte={r['support_rate']}")
    for c in r["claims"]:
        print(f"   {c['label']:10} ent={c['entailment']:.2f} con={c['contradiction']:.2f} | {c['claim'][:70]}")
    for k in r["conflicts"]:
        print(f"   CONFLITO p={k['contradiction']}: '{k['a'][:40]}' x '{k['b'][:40]}'")
    print()
print(f"tempo total: {time.perf_counter()-t0:.1f}s (CPU)")


D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22792.64it/s]

3 trechos recuperados



D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(



Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

... (4 outputs left out)

[fiel] veredito=ok | suporte=1.0
   sustentada ent=0.99 con=0.83 | Segure a manga ou o punho do oponente e puxe o braço dele sobre o seu 
   sustentada ent=0.99 con=0.99 | Passe a perna oposta por cima da cabeça do oponente e aperte os joelho
   sustentada ent=0.80 con=0.90 | Eleve o quadril devagar até o oponente bater.



[contradiz_fonte] veredito=contradiz_fonte | suporte=0.5
   sustentada ent=0.73 con=0.99 | Solte o punho do oponente antes de elevar o quadril.
   contradiz  ent=0.06 con=1.00 | Aplique a chave rápido e com força, sem esperar o oponente bater.



[alucinacao] veredito=pouco_sustentada | suporte=0.0
   neutra     ent=0.11 con=0.21 | O armlock foi criado por Helio Gracie em 1930.
   neutra     ent=0.01 con=0.01 | É a finalização mais usada da história do ADCC.



[incoerente] veredito=contradiz_fonte | suporte=0.5
   sustentada ent=0.74 con=0.35 | Eleve o quadril devagar para finalizar a chave de braço.
   contradiz  ent=0.18 con=1.00 | Nunca eleve o quadril durante a chave de braço.
   CONFLITO p=0.999: 'Eleve o quadril devagar para finalizar a' x 'Nunca eleve o quadril durante a chave de'

tempo total: 13.4s (CPU)


### Achado v1: premissa longa e pares sem relação
Premissa = chunk inteiro → "fiel" saiu `contradiz_fonte` (ent 0,18 no passo 1). Coerência entre todos os pares →
conflito falso entre passos diferentes ("Segure a manga" × "Passe a perna", p=0,95).
Correção: premissas no nível de frase; coerência só entre afirmações com ≥50% de palavras de conteúdo em comum.


## 5.3 Agente final: create_openai_tools_agent + AgentExecutor + verificador

5 tools em 4 categorias (hardcoded, RAG, API, text-to-SQL). A saída passa pelo verificador NLI:
contradiz/incoerente → 1 revisão contra as fontes; pouco sustentada → aviso ao aluno.


In [3]:
from src.agent import build_mestre, TOOL_CATEGORY

mestre = build_mestre()

E_CASES = [
    "Como faço a kimura na guarda fechada?",                                     # RAG
    "Quantos pontos vale uma passagem de guarda?",                              # hardcoded
    "Qual a finalização mais comum da história do ADCC?",                       # text-to-SQL
    "Me mostra um vídeo de raspagem de tesoura e me explica o primeiro passo",  # API + RAG
]

e_results = []
for q in E_CASES:
    t0 = time.perf_counter()
    r = mestre.invoke({"input": q})
    r["question"], r["secs"] = q, round(time.perf_counter() - t0, 1)
    e_results.append(r)
    cats = sorted({TOOL_CATEGORY[t] for t in r["tools"]})
    print(f">>> {q}\n    tools={r['tools']} | categorias={cats}")
    print(f"    veredito={r['verdict']} | suporte={r['support_rate']} | revisada={r['revised']} | {r['secs']}s")
    print(f"    {r['output'][:400]}\n")
    time.sleep(10)  # 8000 tokens/min


>>> Como faço a kimura na guarda fechada?
    tools=['search_technique'] | categorias=['semântica (RAG)']
    veredito=ok | suporte=0.8333333333333334 | revisada=True | 15.8s
    **Kimura na guarda fechada** (passo a passo – até 6 etapas)

1. **Aguarde o momento** – espere o oponente colocar a mão no chão ou empurre o pulso dele para o tapete ao lado do seu quadril.  
2. **Segure o pulso** – agarre esse pulso com a mão do mesmo lado.  
3. **Abra a guarda e crie a pegada** – abra a guarda, sente‑se e passe o braço oposto por cima do ombro dele, pegando seu próprio pulso (pe



>>> Quantos pontos vale uma passagem de guarda?
    tools=['ibjjf_points'] | categorias=['hardcoded']
    veredito=ok | suporte=None | revisada=False | 1.1s
    Uma passagem de guarda vale **3 pontos** segundo as regras da IBJJF.



>>> Qual a finalização mais comum da história do ADCC?
    tools=['query_adcc'] | categorias=['text-to-SQL']
    veredito=ok | suporte=None | revisada=False | 2.5s
    A finalização mais comum na história do ADCC é o **Rear‑Naked Choke (RNC)**, com 99 ocorrências.



>>> Me mostra um vídeo de raspagem de tesoura e me explica o primeiro passo
    tools=['search_technique', 'search_video'] | categorias=['API externa', 'semântica (RAG)']
    veredito=ok | suporte=0.6666666666666666 | revisada=False | 6.7s
    Aqui está um vídeo que demonstra a raspagem de tesoura a partir da guarda fechada:  
**Scissor Sweep for BJJ White Belts (Try These Adjustments)** — https://www.youtube.com/watch?v=UBf7uF5x8GQ  

**Primeiro passo da raspagem de tesoura**  
1️⃣ **Pegada de colar e manga** – Segure o colar do adversário com a mão direita (colocando a mão no colar direito dele) e, ao mesmo tempo, agarre a manga do me



In [4]:
k = e_results[0]
for c in k["check"]["claims"]:
    print(f"{c['label']:10} ent={c['entailment']:.2f} con={c['contradiction']:.2f} | {c['claim'][:110]}")
print("conflitos:", k["check"]["conflicts"])


sustentada ent=0.67 con=0.97 | espere o oponente colocar a mão no chão ou empurre o pulso dele para o tapete ao lado do seu quadril.
sustentada ent=0.64 con=0.94 | agarre esse pulso com a mão do mesmo lado.
sustentada ent=0.92 con=0.83 | abra a guarda, sente‑se e passe o braço oposto por cima do ombro dele, pegando seu próprio pulso (pegada “figu
sustentada ent=0.72 con=0.34 | caia levemente para trás, faça o “shrimp” (deslocamento lateral) em direção ao braço atacado e coloque a perna
sustentada ent=0.63 con=0.97 | mantenha o braço do adversário dobrado em aproximadamente 90 graus e gire lentamente a mão dele em direção às 
neutra     ent=0.01 con=0.63 | aplique pressão até o tap; libere imediatamente ao sentir o tap ou se o adversário começar a sentir dor excess
conflitos: []


### Achado E v1
- Kimura: resposta fiel à ficha, mas NLI marcou `contradiz_fonte` → revisão desnecessária. Causa: formato do gpt-oss
  ("Título – texto" + citação 【...】) poluindo as afirmações. Correção: limpar ambos no `split_claims`.
- Vídeo + 1º passo: agente chamou só `search_video` e explicou o passo de memória; sem trecho do RAG o verificador
  não tinha o que checar e deu `ok`. Correção: `search_technique` obrigatória antes de ensinar + veredito `sem_fonte`.


## 5.4 Memória entre turnos: RunnableWithMessageHistory

Histórico por `session_id` injetado no `chat_history` do prompt do agente. Teste: 3 turnos com referências
implícitas ("isso", "disso") na sessão do aluno 1; a mesma pergunta numa sessão nova não tem contexto.


In [5]:
from src.agent import build_mestre_with_memory, get_session_history

mestre_mem = build_mestre_with_memory()

def ask(q: str, session: str) -> dict:
    r = mestre_mem.invoke({"input": q}, config={"configurable": {"session_id": session}})
    print(f"[{session}] >>> {q}\n    tools={r['tools']} | veredito={r['verdict']}\n    {r['output'][:300]}\n")
    time.sleep(10)
    return r

ask("Sou faixa branca e treino com kimono. Como faço a kimura na guarda fechada?", "aluno-1")
ask("E se ele defender puxando o braço?", "aluno-1")
ask("Me mostra um vídeo disso", "aluno-1")
ask("Me mostra um vídeo disso", "aluno-2")          # sessão nova: sem contexto

print("mensagens no histórico aluno-1:", len(get_session_history("aluno-1").messages))


C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_14288\1737171639.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  mestre_mem = build_mestre_with_memory()
D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\notebooks\..\src\agent.py:148: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  return _SESSIONS.setdefault(session_id, InMemoryChatMessageHistory())


[aluno-1] >>> Sou faixa branca e treino com kimono. Como faço a kimura na guarda fechada?
    tools=['search_technique'] | veredito=pouco_sustentada
    **Kimura na Guarda Fechada (gi)**  

1️⃣ Espere o oponente apoiar a mão no chão ou empurrar o pulso contra o tapete, próximo ao seu quadril.  
2️⃣ Agarre esse pulso com a mão do mesmo lado, mantendo o cotovelo a 90°.  
3️⃣ Abra a guarda, sente‑se e passe o braço oposto por cima do ombro dele; segure



[aluno-1] >>> E se ele defender puxando o braço?
    tools=['search_technique'] | veredito=contradiz_fonte
    **Quando o oponente tenta defender a kimura na guarda fechada**

1️⃣ **Mantenha o cotovelo dele a 90°** – evite que ele estenda o braço; segure o pulso firmemente para impedir a rotação. (fonte: kimura.md – Common mistakes)

2️⃣ **Desloque o quadril (shrimp) para o lado do braço atacado**, criando o



[aluno-1] >>> Me mostra um vídeo disso
    tools=['search_technique', 'search_video'] | veredito=ok
    **Vídeo recomendado:**  
*Kimura From Closed Guard For White Belts (Small Details To Improve Success)* — https://www.youtube.com/watch?v=mVkKOPNGvjA  

Assista para ver a sequência completa, incluindo a pegada figure‑four, o shrimp e o controle da perna. Lembre‑se de treinar devagar, sempre pedindo 



[aluno-2] >>> Me mostra um vídeo disso
    tools=[] | veredito=ok
    Qual técnica ou posição você gostaria de ver em vídeo?



mensagens no histórico aluno-1: 6


### Resultado v1 (mestre-v1)

tool_recall 1,00 · tool_precision 0,88 · quality 0,79 (1 NaN) · verified 0,62

- Triângulo: query sem "steps" recuperou Defenses → agente ensinou a defesa como ataque; NLI marcou contradiz_fonte.
- Berimbolo: pediu posição em vez de admitir que a KB não cobre (quality 0).
- De La Riva: juiz 20b devolveu vazio → OutputParserException (NaN).
  Correções v2: seção na query ('steps'/'defenses'), regra "só mencionada ≠ ensinada", citação normalizada, juiz com fallback 120b.


## 5.6 Avaliação no LangSmith: qualidade + trajetória

Dataset `bjj-mestre-eval` (8 perguntas, todas as categorias de tool + 1 fora da KB). Cada exemplo tem as tools
esperadas e uma resposta de referência. Avaliadores:
- **tool_recall** / **tool_precision** (trajetória): tools esperadas × executadas;
- **quality** (LLM-as-judge, 1–5 normalizado): responde à pergunta, coerente com a referência, sem invenção;
- **verified** (verificador NLI): veredito `ok`.


In [3]:
from langsmith import Client, evaluate

ls_client = Client()
DATASET = "bjj-mestre-eval"

EXAMPLES = [
    ("Como faço o triângulo da guarda fechada?", ["search_technique"],
     "Passo a passo do triângulo a partir da guarda fechada, baseado na ficha triangle_closed_guard.md."),
    ("Como escapo da montada?", ["search_technique"],
     "Fuga upa: prender braço e perna do mesmo lado, ponte e rolar (mount_escape_upa.md)."),
    ("Quantos pontos vale uma montada?", ["ibjjf_points"], "4 pontos (IBJJF)."),
    ("Quantas lutas terminaram em finalização em 2022 no ADCC?", ["query_adcc"], "40 lutas."),
    ("Quem foi Helio Gracie?", ["search_wikipedia"],
     "Artista marcial brasileiro, um dos fundadores do Gracie Jiu-Jitsu."),
    ("Me mostra um vídeo de kimura", ["search_video"], "Um link do YouTube de kimura."),
    ("Me ensina a passar a guarda De La Riva e me mostra um vídeo", ["search_technique", "search_video"],
     "Passos da passagem da De La Riva (dlr_guard_pass.md) + link de vídeo."),
    ("Como faço o berimbolo?", [],
     "Deve admitir que a base de conhecimento não ensina o berimbolo, sem inventar passos."),
]

if not ls_client.has_dataset(dataset_name=DATASET):
    ds = ls_client.create_dataset(DATASET, description="Mestre BJJ: qualidade e trajetória (Fase 5.6)")
    ls_client.create_examples(dataset_id=ds.id, examples=[
        {"inputs": {"input": q}, "outputs": {"expected_tools": tools, "reference": ref}}
        for q, tools, ref in EXAMPLES
    ])
print(DATASET, "→", len(list(ls_client.list_examples(dataset_name=DATASET))), "exemplos")


bjj-mestre-eval → 8 exemplos


In [4]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

mestre_eval = build_mestre()  # sem memória: cada exemplo é independente


def target(inputs: dict) -> dict:
    r = mestre_eval.invoke({"input": inputs["input"]})
    time.sleep(8)  # 8000 tokens/min
    return {"output": r["output"], "tools": r["tools"], "verdict": r["verdict"]}


# --- trajetória ---
def tool_recall(outputs: dict, reference_outputs: dict) -> dict:
    exp, got = set(reference_outputs["expected_tools"]), set(outputs["tools"])
    return {"key": "tool_recall", "score": len(exp & got) / len(exp) if exp else 1.0}


def tool_precision(outputs: dict, reference_outputs: dict) -> dict:
    exp, got = set(reference_outputs["expected_tools"]), set(outputs["tools"])
    return {
        "key": "tool_precision",
        "score": len(exp & got) / len(got) if got else float(not exp),
    }


# --- qualidade (LLM-as-judge) ---
class Grade(BaseModel):
    reasoning: str = Field(description="Short justification")
    score: int = Field(description="1 (bad) to 5 (excellent)")


grade_parser = PydanticOutputParser(pydantic_object=Grade)
judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You grade answers of a Brazilian Jiu-Jitsu tutor. Criteria: answers the question; consistent "
            "with the reference; no invented steps, numbers or links; written in Portuguese. "
            "Citations like (fonte: ...) are expected, not a problem.\n{format_instructions}",
        ),
        ("human", "Question: {question}\nReference: {reference}\nAnswer: {answer}"),
    ]
).partial(format_instructions=grade_parser.get_format_instructions())

# 20b barato; se devolver vazio/JSON inválido, cai para o 120b
judge = (judge_prompt | llm_small | grade_parser).with_fallbacks(
    [judge_prompt | llm | grade_parser]
)


def quality(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    g = judge.invoke(
        {
            "question": inputs["input"],
            "reference": reference_outputs["reference"],
            "answer": outputs["output"],
        }
    )
    return {"key": "quality", "score": (g.score - 1) / 4, "comment": g.reasoning}


# --- fidelidade (verificador NLI) ---
def verified(outputs: dict) -> dict:
    return {
        "key": "verified",
        "score": float(outputs["verdict"] == "ok"),
        "comment": outputs["verdict"],
    }


results = evaluate(
    target,
    data=DATASET,
    evaluators=[tool_recall, tool_precision, quality, verified],
    experiment_prefix="mestre-v2",
    max_concurrency=1,
)

D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'mestre-v2-85876c61' at:
https://smith.langchain.com/o/0f8d36c1-6509-4d30-b196-62b65caf9aaf/datasets/8f4b796f-9680-4b16-8bf5-4e9d785723fa/compare?selectedSessions=b3e8d5ff-6919-445a-9bb9-edfcf2715829





0it [00:00, ?it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12688.18it/s]

D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


... (10 outputs left out)


6it [01:54, 15.53s/it]


7it [02:04, 13.69s/it]


8it [02:05,  9.56s/it]


8it [02:05, 15.64s/it]

In [5]:
df = results.to_pandas()
cols = ["inputs.input", "outputs.tools", "outputs.verdict",
        "feedback.tool_recall", "feedback.tool_precision", "feedback.quality", "feedback.verified"]
display(df[cols])
print(df[[c for c in cols if c.startswith("feedback.")]].mean().round(2))


,inputs.input,outputs.tools,outputs.verdict,feedback.tool_recall,feedback.tool_precision,feedback.quality,feedback.verified
0,Quantos pontos vale uma montada?,[ibjjf_points],ok,1.0,1.0,1.00,1.0
1,Me ensina a passar a guarda De La Riva e me mostra um vídeo,"[search_technique, search_video]",ok,1.0,1.0,1.00,1.0
2,Como escapo da montada?,[search_technique],pouco_sustentada,1.0,1.0,0.50,0.0
3,Como faço o berimbolo?,"[search_technique, search_video]",pouco_sustentada,1.0,0.0,0.25,0.0
4,Como faço o triângulo da guarda fechada?,[search_technique],contradiz_fonte,1.0,1.0,1.00,0.0
5,Quantas lutas terminaram em finalização em 2022 no ADCC?,[query_adcc],ok,1.0,1.0,1.00,1.0
6,Quem foi Helio Gracie?,[search_wikipedia],ok,1.0,1.0,1.00,1.0
7,Me mostra um vídeo de kimura,[search_video],ok,1.0,1.0,1.00,1.0


feedback.tool_recall       1.00
feedback.tool_precision    0.88
feedback.quality           0.84
feedback.verified          0.62
dtype: float64


In [2]:
# simula o ambiente do professor: sem chave, sem rede
os.environ["USE_DUMMY"] = "true"
import importlib, src.tools, src.chains, src.agent
for m in (src.tools, src.chains, src.agent):
    importlib.reload(m)

dummy = src.agent.build_mestre_with_memory()
for q in ["Como faço a kimura?", "E se ele defender?"]:
    r = dummy.invoke({"input": q}, config={"configurable": {"session_id": "prof"}})
    print(r["tools"], r["verdict"], "|", r["output"][:120])
print("histórico:", len(src.agent.get_session_history("prof").messages), "mensagens")
os.environ["USE_DUMMY"] = "false"


C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_16496\4145641453.py:7: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  dummy = src.agent.build_mestre_with_memory()
D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\notebooks\..\src\agent.py:151: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  return _SESSIONS.setdefault(session_id, InMemoryChatMessageHistory())


D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   2%|▏         | 5/202 [00:00<00:04, 41.60it/s]

Loading weights:  49%|████▉     | 99/202 [00:00<00:00, 522.52it/s]

Loading weights:  84%|████████▍ | 170/202 [00:00<00:00, 600.44it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 550.75it/s]

['search_technique'] pouco_sustentada | (dummy) Segundo a base de conhecimento: [armlock_closed_guard.md / Steps] (dummy) Control one arm, pivot, leg over the h


['search_technique'] pouco_sustentada | (dummy) Segundo a base de conhecimento: [armlock_closed_guard.md / Steps] (dummy) Control one arm, pivot, leg over the h
histórico: 4 mensagens
